# Day 1 - setup + letters + INCLUDE keypoints (CPU only)
Add inputs first (right sidebar): `slr-code`, ASL Alphabet (`grassknoted/asl-alphabet`), an ISL fingerspelling image dataset, the pretrained-ASL dataset (`209sontung/sign-language`), and the `asl-signs` competition. Internet ON.

In [ ]:
# Code bundle: upload dist/slr_code.zip as a Kaggle Dataset named "slr-code", then add it as input.
import glob, os, shutil, subprocess, sys
src = (glob.glob("/kaggle/input/slr-code*") or glob.glob("/content/slr_code"))[0]
shutil.copytree(src, "/kaggle/working/code" if os.path.exists("/kaggle") else "/content/code", dirs_exist_ok=True)
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
CODE = f"{WORK}/code"
os.chdir(CODE)
# mediapipe 0.10.14 still has the `mp.solutions` API we use. If pip complains about numpy, run
# `!pip install "numpy<2"` and restart the session once.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe==0.10.14", "remotezip", "opencv-python-headless"])

## 1. Smoke-test the pretrained ASL weights on ONE clip (<= 1 hour; else switch to `abhinand5/isolated-sign-language-recognition`)

In [ ]:
MODEL_DIR = "/kaggle/input/sign-language"   # <- folder of the pretrained model dataset (check the sidebar)
!python scripts/asl_words.py inspect --model-dir $MODEL_DIR
!python scripts/asl_words.py infer --model-dir $MODEL_DIR --data /kaggle/input/asl-signs --signs hello --per-sign 1 --out $WORK/smoke.json

## 2. Letters: ~300 images per letter -> hand landmarks -> classifier (minutes on CPU)

In [ ]:
ASL_IMAGES = "/kaggle/input/asl-alphabet"          # root of the ASL Alphabet dataset (class folder auto-detected)
ISL_IMAGES = "/kaggle/input/REPLACE-WITH-ISL-FINGERSPELLING-DATASET"
for lang, root in [("asl", ASL_IMAGES), ("isl", ISL_IMAGES)]:
    !python scripts/letters.py extract --images $root --lang $lang --out $WORK/letters --per-class 300 --workers 4
    !python scripts/letters.py train --lang $lang --data $WORK/letters --out $WORK/models

## 3. INCLUDE-50 keypoints (reads only the 50 signs' videos from each Zenodo part, deletes each video right after)
Run `--list` first to see the parts. Re-running resumes where it stopped, so a session timeout costs nothing.

In [ ]:
!python scripts/include_extract.py --out $WORK/include50 --list

In [ ]:
!python scripts/include_extract.py --out $WORK/include50 --workers 8 --stride 2
!du -sh $WORK/include50

## 4. Save everything
Commit the notebook (Save Version -> Save & Run All), then Output -> **New Dataset** to keep `models/`, `letters/`, `include50/`.